In [1]:
import random

CUTOFF = "CUTOFF"
FAILURE = "FAILURE"
pop_count = 0

m, n = map(int, input("Nhap so dong va cot: ").split())
room = []

x = random.randint(0, m - 1)
y = random.randint(0, n - 1)
print(f"Vi tri bat dau: {x},{y}")

for i in range(m):
    row = list(map(int, input().split()))
    room.append(row)

room[x][y] = 0

def print_room(room_state, x, y):
    for i in range(m):
        for j in range(n):
            if x == i and y == j:
                print("M", end=" ")
            else:
                print(room_state[i][j], end=" ")
        print()


print("Trang thai bat dau:")
print_room(room, x, y)

def count_trash(room_state):
    return sum(row.count(1) for row in room_state)

def is_clean(room_state):
    return count_trash(room_state) == 0

def get_children(current_room, x, y):
    children = []

    def clean(x, y):
        copy_room = [list(row) for row in current_room]
        copy_room[x][y] = 0
        return tuple(tuple(row) for row in copy_room)

    if x > 0:
        children.append((clean(x - 1, y), x - 1, y, "UP"))
    if x < m - 1:
        children.append((clean(x + 1, y), x + 1, y, "DOWN"))
    if y > 0:
        children.append((clean(x, y - 1), x, y - 1, "LEFT"))
    if y < n - 1:
        children.append((clean(x, y + 1), x, y + 1, "RIGHT"))

    return children

def ida_star_search(start_room, start_x, start_y):
    start_room = tuple(tuple(row) for row in start_room)

    f_limit = count_trash(start_room)

    while True:
        ancestors = frozenset([(start_room, start_x, start_y)])

        result, next_f_limit = f_limited_search(
            start_room, start_x, start_y, 0, [], f_limit, ancestors
        )

        if result != CUTOFF:
            if result == FAILURE:
                return None
            return result

        if next_f_limit == float("inf"):
            return None
        f_limit = next_f_limit


def f_limited_search(current_room, x, y, g_cost, path, f_limit, ancestors):
    global pop_count
    pop_count += 1

    h_cost = count_trash(current_room)
    f_cost = g_cost + h_cost

    if f_cost > f_limit:
        return CUTOFF, f_cost

    if is_clean(current_room):
        return path, f_limit

    min_cutoff = float("inf")
    cutoff_occurred = False

    for next_room, next_x, next_y, action in get_children(current_room, x, y):
        child_signature = (next_room, next_x, next_y)

        if child_signature not in ancestors:
            new_path = path + [action]
            new_ancestors = frozenset(ancestors | {child_signature})

            result, next_f = f_limited_search(
                next_room,
                next_x,
                next_y,
                g_cost + 1,
                new_path,
                f_limit,
                new_ancestors,
            )

            if result != CUTOFF:
                if result != FAILURE:
                    return result, f_limit

            else:
                cutoff_occurred = True
                if next_f < min_cutoff:
                    min_cutoff = next_f

    if cutoff_occurred:
        return CUTOFF, min_cutoff
    return FAILURE, float("inf")


actions = ida_star_search(room, x, y)

print("\n------Ap dung thuat toan IDA* Search------")

if actions is None:
    print("Khong tim thay duong di!")
else:
    print(f"Tim thay giai phap! Tong so buoc di chuyen: {len(actions)}")
    print("Cac buoc thuc hien:", " -> ".join(actions))
    print("\n--- MO PHONG TUNG BUOC CHAY ---")

    current_room_state = [list(row) for row in room]
    curr_x, curr_y = x, y

    for idx, act in enumerate(actions, 1):
        print(f"Buoc {idx}: {act}")
        if act == "UP":
            curr_x -= 1
        elif act == "DOWN":
            curr_x += 1
        elif act == "LEFT":
            curr_y -= 1
        elif act == "RIGHT":
            curr_y += 1

        current_room_state[curr_x][curr_y] = 0
        print_room(current_room_state, curr_x, curr_y)

    print(f"\nTong so lan duyet qua node (pop/visit): {pop_count}")

Vi tri bat dau: 1,2
Trang thai bat dau:
1 1 1 
1 1 M 
1 1 1 

------Ap dung thuat toan IDA* Search------
Tim thay giai phap! Tong so buoc di chuyen: 9
Cac buoc thuc hien: UP -> DOWN -> DOWN -> LEFT -> UP -> UP -> LEFT -> DOWN -> DOWN

--- MO PHONG TUNG BUOC CHAY ---
Buoc 1: UP
1 1 M 
1 1 0 
1 1 1 
Buoc 2: DOWN
1 1 0 
1 1 M 
1 1 1 
Buoc 3: DOWN
1 1 0 
1 1 0 
1 1 M 
Buoc 4: LEFT
1 1 0 
1 1 0 
1 M 0 
Buoc 5: UP
1 1 0 
1 M 0 
1 0 0 
Buoc 6: UP
1 M 0 
1 0 0 
1 0 0 
Buoc 7: LEFT
M 0 0 
1 0 0 
1 0 0 
Buoc 8: DOWN
0 0 0 
M 0 0 
1 0 0 
Buoc 9: DOWN
0 0 0 
0 0 0 
M 0 0 

Tong so lan duyet qua node (pop/visit): 192
